# 05. 글자 읽기(OCR) · QR 코드

- **글자**: `easyocr` (한글 + 영어). 글자 자리 찾기 모델과 글자 읽기 모델 두 개가 이어져 돈다. CPU(PyTorch)로 돈다.
- **QR**: OpenCV 의 `QRCodeDetector` — AI 모델이 아니라 영상 처리 알고리즘이다.

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

## 2. 사진 — 글자가 보이게 찍는다 (책 표지, 안내문 등)

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 글자 읽기 — easyocr

In [ ]:
import easyocr

OCR_DIR = MODELS / "code" / "easyocr"
reader = easyocr.Reader(["ko", "en"], gpu=False,
                        model_storage_directory=str(OCR_DIR), user_network_directory=str(OCR_DIR),
                        download_enabled=False)              # 인터넷으로 받으러 가지 않는다

grey = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)              # 흑백으로 넣는다 (윈도우 일부에서 컬러 경로가 오류)
t0 = time.perf_counter()
results = reader.readtext(grey)                             # [(꼭짓점 4개, 글자, 확신도), ...]
print(f"{len(results)}줄 · {(time.perf_counter() - t0) * 1000:.0f} ms")
for quad, text, conf in results:
    print(f"{conf:.2f}  {text}")

In [ ]:
view = frame.copy()
for quad, text, conf in results:
    pts = np.array(quad, dtype=np.int32)
    cv2.polylines(view, [pts], True, (0, 160, 255), 2)       # 글자 자리는 기울어진 사각형(꼭짓점 4개)
show(view)

## 4. QR — 만들고 읽기
QR 은 만들기도 OpenCV 로 된다. 만든 것을 바로 읽어 보고, 웹캠 사진에서도 찾아본다.

In [ ]:
enc = cv2.QRCodeEncoder.create()
qr = enc.encode("https://example.com/edge-lab")             # 흑백 작은 그림
qr = cv2.resize(qr, (qr.shape[1] * 8, qr.shape[0] * 8), interpolation=cv2.INTER_NEAREST)
qr = cv2.copyMakeBorder(qr, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255)
qr_bgr = cv2.cvtColor(qr, cv2.COLOR_GRAY2BGR)
show(qr_bgr, 240)

def read_qr(bgr):
    ok, texts, pts, _ = cv2.QRCodeDetector().detectAndDecodeMulti(bgr)
    if not ok or pts is None:
        return []
    return [(t, p.astype(int)) for t, p in zip(texts, pts) if t]

print("만든 QR:", [t for t, _ in read_qr(qr_bgr)])
print("웹캠 사진:", [t for t, _ in read_qr(frame)] or "QR 없음")

## 해 볼 것
1. `reader.readtext(grey, detail=0)` 은 무엇을 돌려주나? `paragraph=True` 는?
2. 확신도 0.5 아래는 버리고, 위에서 아래 · 왼쪽에서 오른쪽 순서로 글을 이어 붙여 보자.
3. 사진을 흐리게(`cv2.GaussianBlur`) · 작게(`cv2.resize`) 만들면 어디서부터 못 읽나?
4. QR 에 한글을 넣어 만들고 읽어 보자.